In [1]:
import pandas as pd

df = pd.read_csv('phishing_site_urls.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nLabel counts:")
print(df['Label'].value_counts())

Shape: (549346, 2)

First 5 rows:
                                                 URL Label
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...   bad
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...   bad
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....   bad
3  mail.printakid.com/www.online.americanexpress....   bad
4  thewhiskeydregs.com/wp-content/themes/widescre...   bad

Label counts:
Label
good    392924
bad     156422
Name: count, dtype: int64


In [2]:
import re
from urllib.parse import urlparse

def extract_features(url):
    features = {}
    
    # 1. URL Length
    features['url_length'] = len(url)
    
    # 2. Count of dots
    features['dot_count'] = url.count('.')
    
    # 3. Count of hyphens
    features['hyphen_count'] = url.count('-')
    
    # 4. Count of slashes
    features['slash_count'] = url.count('/')
    
    # 5. Has @ symbol
    features['has_at'] = 1 if '@' in url else 0
    
    # 6. Has IP address
    features['has_ip'] = 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0
    
    # 7. Has HTTPS
    features['has_https'] = 1 if url.startswith('https') else 0
    
    # 8. URL Depth (how many folders deep)
    features['url_depth'] = url.count('/')
    
    # 9. Suspicious keywords
    keywords = ['login', 'verify', 'secure', 'update', 'bank', 'paypal', 'account', 'password']
    features['has_keywords'] = 1 if any(k in url.lower() for k in keywords) else 0
    
    # 10. Domain length
    try:
        domain = urlparse('http://' + url).netloc
        features['domain_length'] = len(domain)
    except:
        features['domain_length'] = 0
    
    return features

# Test it on one URL
sample = df['URL'][0]
print("URL:", sample)
print("\nFeatures:", extract_features(sample))

URL: nobell.it/70ffb52d079109dca5664cce6f317373782/login.SkyPe.com/en/cgi-bin/verification/login/70ffb52d079109dca5664cce6f317373/index.php?cmd=_profile-ach&outdated_page_tmpl=p/gen/failed-to-load&nav=0.5.1&login_access=1322408526

Features: {'url_length': 225, 'dot_count': 6, 'hyphen_count': 4, 'slash_count': 10, 'has_at': 0, 'has_ip': 0, 'has_https': 0, 'url_depth': 10, 'has_keywords': 1, 'domain_length': 9}


In [3]:
print("Extracting features... please wait ⏳")

# Extract features for all URLs
feature_list = []
for url in df['URL']:
    feature_list.append(extract_features(url))

# Convert to dataframe
features_df = pd.DataFrame(feature_list)

# Add label column (convert good/bad to 0/1)
features_df['label'] = df['Label'].map({'good': 0, 'bad': 1})

print("Done! ✅")
print("\nShape:", features_df.shape)
print("\nFirst 5 rows:")
print(features_df.head())

Extracting features... please wait ⏳
Done! ✅

Shape: (549346, 11)

First 5 rows:
   url_length  dot_count  hyphen_count  slash_count  has_at  has_ip  \
0         225          6             4           10       0       0   
1          81          5             2            4       0       0   
2         177          7             1           11       0       0   
3          60          6             0            2       0       0   
4         116          1             1           10       0       0   

   has_https  url_depth  has_keywords  domain_length  label  
0          0         10             1              9      1  
1          0          4             1             15      1  
2          0         11             1             16      1  
3          0          2             0             18      1  
4          0         10             0             19      1  


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle

# Separate features and label
X = features_df.drop('label', axis=1)
y = features_df['label']

# Split data — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training model... please wait ⏳")

# Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Test the model
y_pred = model.predict(X_test)

print("Done! ✅")
print("\nAccuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=['Good', 'Phishing']))

# Save the model
pickle.dump(model, open('phishguard_model.pkl', 'wb'))
print("\nModel saved as phishguard_model.pkl ✅")

Training model... please wait ⏳
Done! ✅

Accuracy: 86.66 %

Detailed Report:
              precision    recall  f1-score   support

        Good       0.88      0.94      0.91     78670
    Phishing       0.81      0.69      0.75     31200

    accuracy                           0.87    109870
   macro avg       0.85      0.81      0.83    109870
weighted avg       0.86      0.87      0.86    109870


Model saved as phishguard_model.pkl ✅


In [1]:
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Use the features_df we already created
X = features_df.drop('label', axis=1)
y = features_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Retrain
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Save model WITH feature names
import joblib
joblib.dump(model, 'phishguard_model.pkl')
print("Model saved correctly ✅")
print("Feature names:", list(X.columns))

NameError: name 'features_df' is not defined

In [2]:
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

# Load dataset
df = pd.read_csv('phishing_site_urls.csv')

# Feature extraction function
def extract_features(url):
    features = {}
    if not url.startswith('http'):
        full_url = 'http://' + url
    else:
        full_url = url
    parsed = urlparse(full_url)
    domain = parsed.netloc
    path = parsed.path
    features['url_length'] = len(url)
    features['dot_count'] = url.count('.')
    features['hyphen_count'] = url.count('-')
    features['slash_count'] = url.count('/')
    features['has_at'] = 1 if '@' in url else 0
    features['has_ip'] = 1 if re.search(r'^\d+\.\d+\.\d+\.\d+', domain) else 0
    features['has_https'] = 1 if url.startswith('https') else 0
    features['url_depth'] = len([x for x in path.split('/') if x])
    keywords = ['login', 'verify', 'secure', 'update', 'bank', 'paypal', 'account', 'password']
    features['has_keywords'] = 1 if any(k in url.lower() for k in keywords) else 0
    features['domain_length'] = len(domain)
    return features

# Extract features for all URLs
print("Extracting features... ⏳")
feature_list = [extract_features(url) for url in df['URL']]
features_df = pd.DataFrame(feature_list)
features_df['label'] = df['Label'].map({'good': 0, 'bad': 1})

# Train model
X = features_df.drop('label', axis=1)
y = features_df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training model... ⏳")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Accuracy
print("Accuracy:", round(accuracy_score(y_test, model.predict(X_test)) * 100, 2), "%")

# Save model
joblib.dump(model, 'phishguard_model.pkl')
print("Model saved correctly ✅")
print("Feature names:", list(X.columns))

Extracting features... ⏳


ValueError: Invalid IPv6 URL

In [3]:
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

# Load dataset
df = pd.read_csv('phishing_site_urls.csv')

# Feature extraction function
def extract_features(url):
    features = {}
    try:
        if not url.startswith('http'):
            full_url = 'http://' + url
        else:
            full_url = url
        parsed = urlparse(full_url)
        domain = parsed.netloc
        path = parsed.path
    except:
        domain = ''
        path = ''

    features['url_length'] = len(url)
    features['dot_count'] = url.count('.')
    features['hyphen_count'] = url.count('-')
    features['slash_count'] = url.count('/')
    features['has_at'] = 1 if '@' in url else 0
    features['has_ip'] = 1 if re.search(r'^\d+\.\d+\.\d+\.\d+', domain) else 0
    features['has_https'] = 1 if url.startswith('https') else 0
    features['url_depth'] = len([x for x in path.split('/') if x])
    keywords = ['login', 'verify', 'secure', 'update', 'bank', 'paypal', 'account', 'password']
    features['has_keywords'] = 1 if any(k in url.lower() for k in keywords) else 0
    features['domain_length'] = len(domain)
    return features

# Extract features for all URLs
print("Extracting features... ⏳")
feature_list = [extract_features(url) for url in df['URL']]
features_df = pd.DataFrame(feature_list)
features_df['label'] = df['Label'].map({'good': 0, 'bad': 1})

# Train model
X = features_df.drop('label', axis=1)
y = features_df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training model... ⏳")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Accuracy
print("Accuracy:", round(accuracy_score(y_test, model.predict(X_test)) * 100, 2), "%")

# Save model
joblib.dump(model, 'phishguard_model.pkl')
print("Model saved correctly ✅")
print("Feature names:", list(X.columns))


Extracting features... ⏳
Training model... ⏳
Accuracy: 87.46 %
Model saved correctly ✅
Feature names: ['url_length', 'dot_count', 'hyphen_count', 'slash_count', 'has_at', 'has_ip', 'has_https', 'url_depth', 'has_keywords', 'domain_length']


In [1]:
import pandas as pd

df = pd.read_csv('malicious_phish.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nLabel counts:")
print(df['type'].value_counts())
print("\nSample URLs:")
print(df.head(10).to_string())


Shape: (651191, 2)

Columns: ['url', 'type']

Label counts:
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

Sample URLs:
                                                                                                                                                                                                                                           url        type
0                                                                                                                                                                                                                             br-icloud.com.br    phishing
1                                                                                                                                                                                                          mp3raid.com/music/krizz_kaliko.html      benign
2                                                         

In [ ]:
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

df = pd.read_csv('malicious_phish.csv')

# Convert to binary — benign=0, everything else=1
df['label'] = df['type'].apply(lambda x: 0 if x == 'benign' else 1)

def extract_features(url):
    features = {}
    try:
        if not url.startswith('http'):
            full_url = 'http://' + url
        else:
            full_url = url
        parsed = urlparse(full_url)
        domain = parsed.netloc
        path = parsed.path
    except:
        domain = ''
        path = ''

    features['url_length'] = len(url)
    features['dot_count'] = url.count('.')
    features['hyphen_count'] = url.count('-')
    features['slash_count'] = url.count('/')
    features['has_at'] = 1 if '@' in url else 0
    features['has_ip'] = 1 if re.search(r'^\d+\.\d+\.\d+\.\d+', domain) else 0
    features['has_https'] = 1 if url.startswith('https') else 0
    features['url_depth'] = len([x for x in path.split('/') if x])
    keywords = ['login', 'verify', 'secure', 'update', 'bank', 'paypal', 'account', 'password']
    features['has_keywords'] = 1 if any(k in url.lower() for k in keywords) else 0
    features['domain_length'] = len(domain)
    features['digit_count'] = sum(c.isdigit() for c in url)
    features['letter_count'] = sum(c.isalpha() for c in url)
    features['special_char_count'] = len(re.findall(r'[^a-zA-Z0-9]', url))
    features['has_subdomain'] = 1 if domain.count('.') > 1 else 0
    features['path_length'] = len(path)
    features['digit_ratio'] = features['digit_count'] / len(url) if len(url) > 0 else 0

    return features

print("Extracting features... ⏳")
feature_list = [extract_features(url) for url in df['url']]
features_df = pd.DataFrame(feature_list)
features_df['label'] = df['label'].values

# Balance dataset
safe = features_df[features_df['label'] == 0].sample(223088, random_state=42)
unsafe = features_df[features_df['label'] == 1]
balanced_df = pd.concat([safe, unsafe]).sample(frac=1, random_state=42)

X = balanced_df.drop('label', axis=1)
y = balanced_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training model... ⏳")
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

print("Accuracy:", round(accuracy_score(y_test, model.predict(X_test)) * 100, 2), "%")
print("\nDetailed Report:")
print(classification_report(y_test, model.predict(X_test), target_names=['Safe', 'Unsafe']))

joblib.dump(model, 'phishguard_model.pkl')
print("\nModel saved ✅")

Extracting features... ⏳
Training model... ⏳


In [1]:
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

df = pd.read_csv('malicious_phish.csv')

# FIX — benign=1 (safe), everything else=0 (unsafe)
df['label'] = df['type'].apply(lambda x: 1 if x == 'benign' else 0)

def extract_features(url):
    features = {}
    try:
        if not url.startswith('http'):
            full_url = 'http://' + url
        else:
            full_url = url
        parsed = urlparse(full_url)
        domain = parsed.netloc
        path = parsed.path
    except:
        domain = ''
        path = ''

    features['url_length'] = len(url)
    features['dot_count'] = url.count('.')
    features['hyphen_count'] = url.count('-')
    features['slash_count'] = url.count('/')
    features['has_at'] = 1 if '@' in url else 0
    features['has_ip'] = 1 if re.search(r'^\d+\.\d+\.\d+\.\d+', domain) else 0
    features['has_https'] = 1 if url.startswith('https') else 0
    features['url_depth'] = len([x for x in path.split('/') if x])
    keywords = ['login', 'veri

SyntaxError: unterminated string literal (detected at line 36) (4001276975.py, line 36)

In [2]:
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

df = pd.read_csv('malicious_phish.csv')

# FIX — benign=1 (safe), everything else=0 (unsafe)
df['label'] = df['type'].apply(lambda x: 1 if x == 'benign' else 0)

def extract_features(url):
    features = {}
    try:
        if not url.startswith('http'):
            full_url = 'http://' + url
        else:
            full_url = url
        parsed = urlparse(full_url)
        domain = parsed.netloc
        path = parsed.path
    except:
        domain = ''
        path = ''

    features['url_length'] = len(url)
    features['dot_count'] = url.count('.')
    features['hyphen_count'] = url.count('-')
    features['slash_count'] = url.count('/')
    features['has_at'] = 1 if '@' in url else 0
    features['has_ip'] = 1 if re.search(r'^\d+\.\d+\.\d+\.\d+', domain) else 0
    features['has_https'] = 1 if url.startswith('https') else 0
    features['url_depth'] = len([x for x in path.split('/') if x])
    keywords = ['login', 'verify', 'secure', 'update', 'bank', 'paypal', 'account', 'password']
    features['has_keywords'] = 1 if any(k in url.lower() for k in keywords) else 0
    features['domain_length'] = len(domain)
    features['digit_count'] = sum(c.isdigit() for c in url)
    features['letter_count'] = sum(c.isalpha() for c in url)
    features['special_char_count'] = len(re.findall(r'[^a-zA-Z0-9]', url))
    features['has_subdomain'] = 1 if domain.count('.') > 1 else 0
    features['path_length'] = len(path)
    features['digit_ratio'] = features['digit_count'] / len(url) if len(url) > 0 else 0
    return features

print("Extracting features... ⏳")
feature_list = [extract_features(url) for url in df['url']]
features_df = pd.DataFrame(feature_list)
features_df['label'] = df['label'].values

# Balance
safe = features_df[features_df['label'] == 1].sample(223088, random_state=42)
unsafe = features_df[features_df['label'] == 0]
balanced_df = pd.concat([safe, unsafe]).sample(frac=1, random_state=42)

X = balanced_df.drop('label', axis=1)
y = balanced_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training model... ⏳")
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

print("Accuracy:", round(accuracy_score(y_test, model.predict(X_test)) * 100, 2), "%")

joblib.dump(model, 'phishguard_model.pkl')
print("Model saved ✅")


Extracting features... ⏳
Training model... ⏳
Accuracy: 95.62 %
Model saved ✅


In [1]:
import pandas as pd

df = pd.read_csv('malicious_phish.csv')

print("Sample BENIGN URLs:")
print(df[df['type'] == 'benign']['url'].head(20).to_string())

print("\nAverage length:")
print("Benign:", df[df['type'] == 'benign']['url'].str.len().mean())
print("Phishing:", df[df['type'] == 'phishing']['url'].str.len().mean())

Sample BENIGN URLs:
1                   mp3raid.com/music/krizz_kaliko.html
2                       bopsecrets.org/rexroth/cr/1.htm
5     http://buzzfil.net/m/show-art/ils-etaient-loin...
6         espn.go.com/nba/player/_/id/3457/brandon-rush
7        yourbittorrent.com/?q=anthony-hamilton-soulife
9         allmusic.com/album/crazy-from-the-heat-r16990
10    corporationwiki.com/Ohio/Columbus/frank-s-bens...
12                       myspace.com/video/vid/30602581
16         quickfacts.census.gov/qfd/maps/iowa_map.html
17    nugget.ca/ArticleDisplay.aspx?archive=true&e=1...
18       uk.linkedin.com/pub/steve-rubenstein/8/718/755
20     baseball-reference.com/players/h/harrige01.shtml
22                  192.com/atoz/people/oakley/patrick/
23    nytimes.com/1998/03/29/style/cuttings-oh-that-...
24                    escholarship.org/uc/item/5xt4952c
25                    songfacts.com/detail.php?id=13410
26                       casamanana.org/education/blba/
27    http://hollywoodlife.c

In [ ]:
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

df = pd.read_csv('malicious_phish.csv')
df['label'] = df['type'].apply(lambda x: 0 if x == 'benign' else 1)

def extract_features(url):
    features = {}
    try:
        if not url.startswith('http'):
            full_url = 'http://' + url
        else:
            full_url = url
        parsed = urlparse(full_url)
        domain = parsed.netloc
        path = parsed.path
    except:
        domain = ''
        path = ''

    features['url_length'] = len(url)
    features['dot_count'] = url.count('.')
    features['hyphen_count'] = url.count('-')
    features['slash_count'] = url.count('/')
    features['has_at'] = 1 if '@' in url else 0
    features['has_ip'] = 1 if re.search(r'^\d+\.\d+\.\d+\.\d+', domain) else 0
    features['has_https'] = 1 if url.startswith('https') else 0
    features['url_depth'] = len([x for x in path.split('/') if x])
    keywords = ['login', 'verify', 'secure', 'update', 'bank', 'paypal', 'account', 'password']
    features['has_keywords'] = 1 if any(k in url.lower() for k in keywords) else 0
    features['domain_length'] = len(domain)
    features['digit_count'] = sum(c.isdigit() for c in url)
    features['letter_count'] = sum(c.isalpha() for c in url)
    features['special_char_count'] = len(re.findall(r'[^a-zA-Z0-9]', url))
    features['has_subdomain'] = 1 if domain.count('.') > 1 else 0
    features['path_length'] = len(path)
    features['digit_ratio'] = features['digit_count'] / len(url) if len(url) > 0 else 0
    return features

print("Extracting features... ⏳")
feature_list = [extract_features(url) for url in df['url']]
features_df = pd.DataFrame(feature_list)
features_df['label'] = df['label'].values

X = features_df.drop('label', axis=1)
y = features_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training model... ⏳")
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

print("Accuracy:", round(accuracy_score(y_test, model.predict(X_test)) * 100, 2), "%")
joblib.dump(model, 'phishguard_model.pkl')
print("Model saved ✅")

Extracting features... ⏳
Training model... ⏳
